In [62]:
# Import Libraries
import numpy as np
import torch

In [63]:
# Goal of this notebook is to learn the very basics of pytorch:
# What a tensor is in pytorch, 

"""
KEY FUNCTIONS
.storage() --> The actual flat block of memory — just a 1D array of numbers
.shape / .size()  --> The dimensions (like MATLAB's size(A))
.stride()  --> How many elements to skip in storage to move one step along each dimension
.dtype  --> Data type (float32, int64, etc.)
.device  --> Where it lives — cpu or cuda:0 (GPU)
.requires_grad  --> Whether to track operations for backprop
.grad_fn  --> The function that created this tensor (the autograd graph link)
.grad  --> Where gradients accumulate after .backward()
.contiguous() --> .contiguous() copies to a fresh sequential layout

""" 


"\nKEY FUNCTIONS\n.storage() --> The actual flat block of memory — just a 1D array of numbers\n.shape / .size()  --> The dimensions (like MATLAB's size(A))\n.stride()  --> How many elements to skip in storage to move one step along each dimension\n.dtype  --> Data type (float32, int64, etc.)\n.device  --> Where it lives — cpu or cuda:0 (GPU)\n.requires_grad  --> Whether to track operations for backprop\n.grad_fn  --> The function that created this tensor (the autograd graph link)\n.grad  --> Where gradients accumulate after .backward()\n.contiguous() --> .contiguous() copies to a fresh sequential layout\n\n"

1: Creating Tensors

In [64]:
# CREATING TENSORS
# From data (like MATLAB literals)
a = torch.tensor([1.0, 2.0, 3.0])
B = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32)

# Allocation (like zeros(), ones(), rand() in MATLAB)
torch.zeros(3, 4)
torch.ones(2, 3)
torch.randn(3, 4)        # normal distribution
torch.rand(3, 4)         # uniform [0, 1)
print( "torch.arange(0, 10, 2)", torch.arange(0, 10, 2))   # like MATLAB 0:2:8

torch.linspace(0, 1, 5)  # like MATLAB linspace

# From NumPy (shared memory — no copy!)
import numpy as np
n = np.array([1.0, 2.0])
t = torch.from_numpy(n)  # mutating t mutates n
print( n )
print( t )
t[0] = 0.0
print( n ) # Note that changing t changes n

torch.arange(0, 10, 2) tensor([0, 2, 4, 6, 8])
[1. 2.]
tensor([1., 2.], dtype=torch.float64)
[0. 2.]


**Reshaping & Dimension Manipulation**
These are your bread and butter. The mental model: you're reinterpreting the same flat storage with different strides/shapes.

In [65]:
# Create a tensor
X = torch.arange(24)
print("X pre-reshape: ", X)
print(X.stride())

print()
X = X.reshape(2,3,4)
#print(X.storage())
print(X)
print( "X stride: ", X.stride())
print("X shape: ", X.shape)
print(X.size())

Y = X.T

print( "Y stride: ", Y.stride() )
#print( " Y storage: ", Y.storage() ) # Same memory
print( " Y contiguous: ", Y.is_contiguous() )


X pre-reshape:  tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23])
(1,)

tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])
X stride:  (12, 4, 1)
X shape:  torch.Size([2, 3, 4])
torch.Size([2, 3, 4])
Y stride:  (1, 4, 12)
 Y contiguous:  False


.view() requires contiguous memory. .reshape() will silently copy if needed. When you're doing thousands of reshapes in a training loop, knowing when copies happen is the difference between fast and OOM.

In [66]:
# Reshape tensors
X = torch.randn(2, 3, 4)  # shape (2, 3, 4)
print( "Original X", X)
print( "Dim", X.shape)

# Reshape — reinterpret shape (may copy if non-contiguous)
X = X.reshape(6, 4)
print( "Reshaped X", X)
print( "Dim", X.shape)
X = X.reshape(-1)          # flatten, like MATLAB's A(:)
print( "Flattened X", X)
print( "Dim", X.shape)
X = X.reshape(2,3,4)

# View -- This explicitly does not copy - fails if non-contiguous
print( "view as 2x12:", X.view(2,12) )

print( "X transposed (0,2) ", X.transpose( 0,2 )) 
print( "X transpose shape ", X.transpose( 0,2 ).shape )

# Permute dimensions
print( "X Permuted (2,0,1) ", X.permute( 2,0,1 )) 
print( "X Permuted shape ", X.permute( 2,0,1 ).shape )


Original X tensor([[[ 0.7323,  0.9053, -0.3172,  1.5200],
         [-1.1908, -0.5645, -0.7120,  1.2677],
         [-0.6072,  0.0209, -1.0635, -2.5057]],

        [[-1.5590, -0.4660, -1.4786, -0.8534],
         [-0.3735,  0.1368,  0.9982,  0.8998],
         [ 0.6178,  0.6570,  0.2644,  0.0839]]])
Dim torch.Size([2, 3, 4])
Reshaped X tensor([[ 0.7323,  0.9053, -0.3172,  1.5200],
        [-1.1908, -0.5645, -0.7120,  1.2677],
        [-0.6072,  0.0209, -1.0635, -2.5057],
        [-1.5590, -0.4660, -1.4786, -0.8534],
        [-0.3735,  0.1368,  0.9982,  0.8998],
        [ 0.6178,  0.6570,  0.2644,  0.0839]])
Dim torch.Size([6, 4])
Flattened X tensor([ 0.7323,  0.9053, -0.3172,  1.5200, -1.1908, -0.5645, -0.7120,  1.2677,
        -0.6072,  0.0209, -1.0635, -2.5057, -1.5590, -0.4660, -1.4786, -0.8534,
        -0.3735,  0.1368,  0.9982,  0.8998,  0.6178,  0.6570,  0.2644,  0.0839])
Dim torch.Size([24])
view as 2x12: tensor([[ 0.7323,  0.9053, -0.3172,  1.5200, -1.1908, -0.5645, -0.7120,  1.267

In [67]:
# Squeeze / Un-squeeze
a = torch.randn(3)
print( "A", a )
print( "Shape: ", a.shape )
print()

print( "a Unsqueezed(0): ", a.unsqueeze(0) )         # shape (1, 3) — like MATLAB row vector
print( a.unsqueeze(0) )
print(a.unsqueeze(0).shape)         # shape (3, 1) — like MATLAB column vector

print()
print( "a Unsqueezed(1): ", a.unsqueeze(1) )         # shape (1, 3) — like MATLAB row vector
print( a.unsqueeze(1) )
print(a.unsqueeze(1).shape)

A tensor([ 2.3399, -1.3675, -1.0583])
Shape:  torch.Size([3])

a Unsqueezed(0):  tensor([[ 2.3399, -1.3675, -1.0583]])
tensor([[ 2.3399, -1.3675, -1.0583]])
torch.Size([1, 3])

a Unsqueezed(1):  tensor([[ 2.3399],
        [-1.3675],
        [-1.0583]])
tensor([[ 2.3399],
        [-1.3675],
        [-1.0583]])
torch.Size([3, 1])


In [68]:
# Expand / repeat — broadcast without copying
a = torch.randn(1, 4)
print( a )

print( a.expand(3, 4) )         # shape (3, 4), no new memory (stride 0 trick)
print(a)

tensor([[-0.7549, -0.1348, -1.5032,  2.1749]])
tensor([[-0.7549, -0.1348, -1.5032,  2.1749],
        [-0.7549, -0.1348, -1.5032,  2.1749],
        [-0.7549, -0.1348, -1.5032,  2.1749]])
tensor([[-0.7549, -0.1348, -1.5032,  2.1749]])


In [69]:
# Indexing

X = torch.randn(4, 5)
print(X)
print()


print("X[0] ", X[0] )         # first row
print() 


print( "X[:, 0] ", X[:, 0] )       # first column
print() 

print( "X[1:3, 2:4] ", X[1:3, 2:4])   # slice
print() 

print( "X[X > 0] ", X[X > 0] )
X[X > 0]      # boolean mask (like MATLAB logical indexing)
print() 

# Advanced indexing (gathers specific elements)
print( "idx = torch.tensor([0, 2, 3])")
idx = torch.tensor([0, 2, 3])
print( idx)
print() 

print( "X[idx]", X[idx] )        # rows 0, 2, 3


tensor([[ 0.4030,  0.0442, -2.3276, -1.0760, -0.2962],
        [-0.6189, -0.6154, -1.6277,  0.5088,  0.7458],
        [-1.8415,  0.2179,  0.5035, -0.5340, -0.9914],
        [-1.5115, -0.2443, -0.8957,  1.0201, -0.3659]])

X[0]  tensor([ 0.4030,  0.0442, -2.3276, -1.0760, -0.2962])

X[:, 0]  tensor([ 0.4030, -0.6189, -1.8415, -1.5115])

X[1:3, 2:4]  tensor([[-1.6277,  0.5088],
        [ 0.5035, -0.5340]])

X[X > 0]  tensor([0.4030, 0.0442, 0.5088, 0.7458, 0.2179, 0.5035, 1.0201])

idx = torch.tensor([0, 2, 3])
tensor([0, 2, 3])

X[idx] tensor([[ 0.4030,  0.0442, -2.3276, -1.0760, -0.2962],
        [-1.8415,  0.2179,  0.5035, -0.5340, -0.9914],
        [-1.5115, -0.2443, -0.8957,  1.0201, -0.3659]])


In [70]:
# Element-wise manipulations (like MATLAB .* ./ .^)
A = torch.randn(2,3)
B = torch.randn(2,3)
print( "A + B: ", A + B )
print("A * B ", A * B )         # element-wise multiply
print( "A ** 2 ", A ** 2 )         # element-wise power
print( "torch.exp(A)", torch.exp(A) )
print( "torch.log(A) ", torch.log(A))
print( "torch.relu(A)", torch.relu(A))  # max(0, x) element-wise

A + B:  tensor([[-0.3705,  1.5301, -0.0224],
        [-2.5221, -1.3102,  0.5286]])
A * B  tensor([[-0.2700,  0.3917, -0.2385],
        [ 1.5900,  0.3929, -0.1561]])
A ** 2  tensor([[0.5429, 1.4521, 0.2497],
        [1.6339, 0.7149, 0.0445]])
torch.exp(A) tensor([[0.4786, 3.3369, 0.6067],
        [0.2785, 0.4293, 0.8098]])
torch.log(A)  tensor([[   nan, 0.1865,    nan],
        [   nan,    nan,    nan]])
torch.relu(A) tensor([[0.0000, 1.2050, 0.0000],
        [0.0000, 0.0000, 0.0000]])


In [71]:
# Matrix multiply (like MATLAB A * B or A @ B)
A = torch.randn(2,3)
B = torch.randn(3,2)
print( A @ B )                  # preferred syntax
print( torch.matmul(A, B) )      # same thing
print( torch.mm(A, B) )         # strictly 2D only

tensor([[-0.1066, -0.1317],
        [ 1.1572, -0.0889]])
tensor([[-0.1066, -0.1317],
        [ 1.1572, -0.0889]])
tensor([[-0.1066, -0.1317],
        [ 1.1572, -0.0889]])


In [72]:
# Batched matmul — this is critical for attention
# (batch, n, m) @ (batch, m, p) → (batch, n, p)
# bmm is strictly "multiply many 2D matrices in parallel." It only
# accepts 3D tensors where the first dimension is the batch.
# The actual multiplication always happens between the last two 
# dimensions (the 2D matrix part). The batch dimension is just
# "do this N times independently."
A = torch.randn(2,3,1)
B = torch.randn(2,1,3)

torch.bmm(A, B)

# second demonstration with bigger matrices.
A = torch.randn(32, 10, 5)   # 32 matrices, each 10×5
B = torch.randn(32, 5, 8)    # 32 matrices, each 5×8

C = torch.bmm(A, B)          # 32 matrices, each 10×8
print(C)

tensor([[[ 2.8348,  1.3963,  1.1015,  ...,  3.7556,  0.0157, -1.8867],
         [ 0.1555, -1.4725,  3.3710,  ..., -2.7073,  2.0409,  1.4245],
         [-0.7260,  0.8561, -0.8837,  ..., -0.7778,  0.2021,  0.5856],
         ...,
         [ 0.6519, -0.0624, -1.2569,  ...,  4.3604, -0.6475, -2.2394],
         [-5.5293,  0.1815, -3.5907,  ..., -4.8360,  1.1499, -2.7377],
         [ 0.8033,  0.1420, -1.5472,  ...,  3.0997, -1.7845, -0.9770]],

        [[ 1.8138, -1.2567, -0.3872,  ...,  1.2718, -2.1820,  2.1961],
         [ 2.7701, -2.7613, -5.5678,  ...,  0.9660,  1.0134,  0.8488],
         [-4.3098,  1.8086,  4.0257,  ..., -1.9245, -1.8093, -0.8432],
         ...,
         [-2.0344,  0.3645,  0.3147,  ...,  1.0592,  3.7527, -4.3296],
         [-0.0734,  2.8616, -0.2151,  ..., -0.5895, -1.0577, -0.7236],
         [-0.2155, -0.5549, -2.4260,  ..., -1.1342, -1.6257,  1.0931]],

        [[-1.6633, -1.5023, -1.4683,  ...,  0.0917,  0.0663,  2.6707],
         [-1.1876,  0.3866, -1.0988,  ..., -0

In [73]:
# Einstein summation — the power tool
torch.einsum('ij,jk->ik', A, B)       # matmul
torch.einsum('bij,bkj->bik', Q, K)    # batched dot product (attention)


RuntimeError: einsum(): the number of subscripts in the equation (2) does not match the number of dimensions (3) for operand 0 and no ellipsis was given

In [ ]:
# Reductions (like MATLAB sum, mean, max along a dim)
X.sum(dim=0)       # sum along rows → shape (5,)
X.mean(dim=1)      # mean along columns → shape (4,)
X.max(dim=1)       # returns (values, indices) tuple